# recurrentlens 02 — Explore a pretrained SAE

Download a community-trained SAE from the Hub and inspect its features.

> **Note (v0.1.0):** Pretrained Mamba-130M SAEs ship in v0.1.1 (trained via notebook 03 on Colab T4). For v0.1.0 you can substitute a locally trained SAE by replacing `REPO_ID` below.

In [ ]:
%pip install -q recurrentlens[mamba] || %pip install -q git+https://github.com/hinanohart/recurrentlens.git

In [ ]:
import torch
import recurrentlens as rl

REPO_ID = 'hinanohart/recurrentlens-mamba130m-L6-sae'  # v0.1.1+
# Until v0.1.1 is up, set REPO_ID = None and skip the load_sae cell.

In [ ]:
# Load model + SAE
model = rl.load_model('state-spaces/mamba-130m-hf')
sae = rl.hub.load_sae(REPO_ID) if REPO_ID else None
print(sae)

In [ ]:
# Collect activations from a small corpus.
from recurrentlens.sae.cache import ActivationCache
from recurrentlens.hooks.registry import HookManager

texts = [
    'The capital of France is Paris.',
    'Quicksort partitions an array around a pivot.',
    'Mamba uses selective state-space mixing.',
    'In thermodynamics, entropy never decreases.',
]
cache = ActivationCache(capacity=4096, d_in=model.d_model, backend='ram')
ctx_per_token = []
with HookManager(model) as mgr:
    h = mgr.add(site=sae.hook_site, layer=sae.layer, transform=lambda t: t.detach().cpu())
    for t in texts:
        ids = model.tokenizer(t, return_tensors='pt').input_ids.to(model.device)
        with torch.no_grad():
            _ = model.forward(ids)
        for tok_idx, tok_id in enumerate(ids[0].tolist()):
            ctx_per_token.append(model.tokenizer.decode([tok_id]))
        cache.append(h.activations[-1].reshape(-1, model.d_model).numpy())
        h.clear()

In [ ]:
# Render top-k activating tokens for a feature.
import torch
acts = torch.as_tensor(cache._store[: len(cache)])
out = rl.viz.feature_explorer(sae, feature_id=42, activations=acts, contexts=ctx_per_token, top_k=10)
out  # renders inline in Jupyter

In [ ]:
# Evaluate the SAE on these activations.
report = rl.bench.evaluate(sae, activations=acts)
print(report)

In [ ]:
# Steer a generation by amplifying feature 42.
ids = model.tokenizer('In the year 2030,', return_tensors='pt').input_ids.to(model.device)
with rl.steer(model, sae, feature_id=42, vector_scale=3.0):
    out = model.hf_model.generate(ids, max_new_tokens=20)
print(model.tokenizer.decode(out[0]))